# ⚠️ 참고용 코드 (여기서는 실행하지 마세요)

이 노트북은 실제 GPU 서버에서 파인튜닝할 때 쓴 코드 그대로입니다. **일반 노트북/PC(CPU만 있는 환경)에서
전체 학습을 돌리면 몇 시간~며칠씩 걸릴 수 있어서 여기서는 실행하지 않는 걸 권장합니다.** 방법론을
보여주기 위한 참고 자료입니다.

실제 결과(파인튜닝된 모델)는 `model/trocr_v3/`에 있고, `1_기본OCR.ipynb`와 `4_이미지업로드확인.ipynb`에서
베이스 모델과 비교해볼 수 있습니다.

---


# 3단계 v2: TrOCR-KO 파인튜닝 (KORIE 데이터 추가)

[MahmoudSalah/KORIE](https://github.com/MahmoudSalah/KORIE) 데이터셋이 저희 `train/images`와
**동일한 원본 이미지**를 쓴다는 걸 확인했습니다(MD5 체크섬 일치). KORIE의 OCR 태스크 필드 단위 크롭
(사람이 정교하게 라벨링한 것으로 추정, 총 8,927개)을 저희 자체 정렬 크롭(5,371개)에 더해 학습 데이터를
늘렸습니다.

- 데이터 (KORIE 원본 구조를 그대로 유지 — `DATA/` 폴더 구조는 README 6절 참고):
  - `DATA/own/metadata_train.csv` (5,371개) — 저희 자체 EasyOCR 검출 + 정답 정렬로 만든 줄 단위 크롭
  - `DATA/KORIE/ocr/{train,val,test}/` (8,927개, `IMG#####_필드명.jpg`+`.txt` 쌍) — KORIE 원본 그대로
  - `DATA/own/metadata_val.csv` (566개, 기존과 동일 — v1과 공정하게 비교하기 위해 검증셋은 바꾸지 않음)
  - 학습 시 두 소스를 **아래 셀에서 그때그때 합쳐서** 사용(사전에 하나로 합친 CSV를 저장해두지 않음 —
    각 소스가 어디서 왔는지 명확하게 유지하기 위함)
- **데이터 누수 방지**: KORIE는 자체 train/val/test가 이미지 단위로 분리되어 있지 않아서(같은 영수증의
  다른 필드가 여러 split에 흩어짐), 3개 split을 전부 합친 뒤 저희 검증 이미지(45장)에 해당하는 KORIE
  크롭은 어느 split에 있든 전부 제외합니다(→ 8,384개 사용).
- 학습: HuggingFace `Seq2SeqTrainer`로 전체 파라미터 파인튜닝
- 참고: 이전(v1, 저희 데이터 5,371개만 사용) 결과 — CER 25.95%→7.88%, WER 42.04%→15.08%


## 0. 환경 설정

In [ ]:
import os

# KoOCR 실험 때문에 이 venv에 TensorFlow도 깔려 있는데, transformers가 TF까지 같이 로드하려다
# (Keras 3 비호환으로) 에러를 내므로, PyTorch만 쓰도록 명시한다.
os.environ["USE_TF"] = "0"
# dataloader_num_workers>0이면 매 에폭마다 워커 프로세스를 fork하는데, 그때마다
# "tokenizers forked..." 경고가 반복 출력됨(무해하지만 시끄러움) -> 미리 비활성화
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import time
import logging

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
import jiwer

from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)

# processing_class=processor를 넘겨도 Trainer 내부 코드가 여전히 구버전 방식(self.tokenizer)으로
# 접근하는 곳이 있어서 "Trainer.tokenizer is now deprecated..." 경고가 스텝마다 반복 출력됨.
# 우리 코드로는 못 없애는 라이브러리 내부 동작이라 로깅 레벨로 억제.
logging.getLogger("transformers.trainer").setLevel(logging.ERROR)

PROJECT_DIR = os.getcwd()
DATA_DIR = os.path.join(PROJECT_DIR, "DATA")

# ============================================================
# V1~V4 비교용 실험 설정. VERSION_TAG를 바꿔서 실행하면 결과가
# trocr_finetuned_v2_experiments/<VERSION_TAG>/ 에 각각 따로 저장되어
# 이전 실행을 덮어쓰지 않음 (06_compare_versions.ipynb에서 모아서 비교).
#
#   V2: VERSION_TAG="v2_no_aug_no_filter", AUGMENT=False, APPLY_HEIGHT_FILTER=False
#   V3: VERSION_TAG="v3_aug_no_filter",    AUGMENT=True,  APPLY_HEIGHT_FILTER=False
#   V4: VERSION_TAG="v4_aug_filter",       AUGMENT=True,  APPLY_HEIGHT_FILTER=True  (이미 학습 완료 - trocr_finetuned_v2/best)
#
# V2, V3는 아직 안 돌렸으니 위 값 중 하나로 바꿔서 실행하세요.
# ============================================================
VERSION_TAG = "v2_no_aug_no_filter"
AUGMENT = False
APPLY_HEIGHT_FILTER = False

OUT_DIR = os.path.join(PROJECT_DIR, "trocr_finetuned_v2_experiments", VERSION_TAG)
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_ID = "team-lucid/trocr-small-korean"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# None이면 전체 데이터, 정수를 넣으면 그 수만큼만(빠른 디버깅용)
SUBSET_N = None
# 노이즈 크롭 제거 + augmentation을 같이 넣은 뒤 val CER이 11.2~11.9% 구간에서 튀며(단조감소 아님)
# patience=2로는 너무 일찍(6에폭) 멈춰서 수렴 전에 종료됨 -> 여유를 늘림.
# V2/V3/V4 모두 같은 설정으로 통일해서 augment/filter 유무만 순수하게 비교되게 함.
NUM_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 4
# 속도 최적화: 배치 32 + 데이터로더 워커 8개(GPU가 67%까지만 차던 건 이미지 전처리가
# 단일 프로세스라 CPU 쪽이 병목이었기 때문 — DATALOADER_NUM_WORKERS로 완화)
BATCH_SIZE = 32
DATALOADER_NUM_WORKERS = 8
LEARNING_RATE = 5e-5
MAX_LABEL_LENGTH = 64

print(f"VERSION_TAG={VERSION_TAG} | AUGMENT={AUGMENT} | APPLY_HEIGHT_FILTER={APPLY_HEIGHT_FILTER}")
print("OUT_DIR:", OUT_DIR)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. 데이터셋

In [ ]:
import glob
import random
from PIL import ImageFilter

OWN_DIR = os.path.join(DATA_DIR, "own")
KORIE_OCR_DIR = os.path.join(DATA_DIR, "KORIE", "ocr")
processor = TrOCRProcessor.from_pretrained(MODEL_ID)
MAX_CROP_HEIGHT = 150


def pil_gaussian_blur(img):
    """torchvision.GaussianBlur(conv2d 기반)는 이 ARM64(GB10) 환경에서 데이터로더 워커
    프로세스 안에서 JIT 백엔드(oneDNN/Xbyak, 원래 x86용) 오류로 크래시남
    ("bad err=3 in Xbyak::Error" -> "label is too far"). PIL 자체 블러 구현은 이 경로를
    안 타서 문제없이 동작 - 멀티워커 스트레스 테스트로 재현/수정 확인함."""
    return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 1.2)))


# 새 매장/폰트/촬영 조건의 영수증에서 성능이 떨어지는 걸 완화하기 위한 약한 augmentation.
# 글자가 안 읽힐 정도로 세게는 안 넣음(과한 회전/왜곡은 정답 자체를 훼손함) - train에만 적용, val엔 미적용.
train_transform = transforms.Compose([
    transforms.RandomApply([transforms.RandomRotation(degrees=3)], p=0.5),
    transforms.RandomApply([transforms.Lambda(pil_gaussian_blur)], p=0.3),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomApply(
        [transforms.RandomAffine(degrees=0, translate=(0.02, 0.02), scale=(0.95, 1.05), shear=3)], p=0.3
    ),
])


class LineDataset(Dataset):
    """df는 최소한 abs_path(이미지 절대경로), text(정답) 컬럼을 가져야 함."""

    def __init__(self, df, processor, transform=None, max_label_length=MAX_LABEL_LENGTH):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.transform = transform
        self.max_label_length = max_label_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["abs_path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze(0)

        labels = self.processor.tokenizer(
            row["text"],
            padding="max_length",
            max_length=self.max_label_length,
            truncation=True,
        ).input_ids
        labels = [l if l != self.processor.tokenizer.pad_token_id else -100 for l in labels]

        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}


def filter_by_height(df, max_height=MAX_CROP_HEIGHT):
    """세로로 비정상적으로 긴 크롭(여러 줄/바코드 등이 섞여 들어간 경우) 제외."""
    if not APPLY_HEIGHT_FILTER:
        return df, 0
    heights = df["abs_path"].apply(lambda p: Image.open(p).height)
    keep = heights <= max_height
    return df[keep].reset_index(drop=True), int((~keep).sum())


# --- 1) 우리 자체 데이터 (EasyOCR 검출 + 정답 정렬로 만든 줄 단위 크롭) ---
df_own_train = pd.read_csv(os.path.join(OWN_DIR, "metadata_train.csv"))
df_own_train["abs_path"] = df_own_train["file_path"].apply(lambda p: os.path.join(OWN_DIR, p))

df_val = pd.read_csv(os.path.join(OWN_DIR, "metadata_val.csv"))
df_val["abs_path"] = df_val["file_path"].apply(lambda p: os.path.join(OWN_DIR, p))
# val은 v1/v2/v3/v4와 계속 비교해야 하므로 그대로 둠(필터 미적용) - 학습 데이터만 정리한다.

# 검증 이미지(45장)는 KORIE 쪽에서도 어느 split에 있든 제외해야 하므로, 파일명에서 이미지 ID를 추출
val_image_ids = set(df_val["file_path"].str.extract(r"(IMG\d+)")[0])

# --- 2) KORIE OCR 태스크 크롭 (사람이 라벨링한 필드 단위 크롭, 원본 구조 그대로 DATA/KORIE/ocr/ 에 보관) ---
# KORIE 자체 train/val/test는 이미지 단위로 분리되어 있지 않으므로(같은 영수증이 여러 split에 흩어짐),
# 3개 split을 모두 합친 뒤 우리 검증 이미지에 해당하는 크롭만 제외한다.
korie_rows = []
for split in ["train", "val", "test"]:
    for txt_path in glob.glob(os.path.join(KORIE_OCR_DIR, split, "*.txt")):
        base = os.path.splitext(os.path.basename(txt_path))[0]
        image_id = base.split("_")[0]
        if image_id in val_image_ids:
            continue
        text = open(txt_path, encoding="utf-8").read().strip()
        if not text:
            continue
        img_path = os.path.join(KORIE_OCR_DIR, split, base + ".jpg")
        if not os.path.exists(img_path):
            continue
        korie_rows.append({"abs_path": img_path, "text": text})

df_korie_train = pd.DataFrame(korie_rows)

# --- 2-1) 비정상적으로 큰 크롭 제외 (표본 검수 결과: 인접 줄/바코드가 섞여 들어간 노이즈 크롭) ---
# APPLY_HEIGHT_FILTER=False면 아무것도 걸러내지 않음 (V2/V3 재현용)
df_own_train, n_dropped_own = filter_by_height(df_own_train)
df_korie_train, n_dropped_korie = filter_by_height(df_korie_train)

# --- 3) 학습 데이터 = own train + KORIE(검증 이미지 제외) ---
df_train = pd.concat(
    [df_own_train[["abs_path", "text"]], df_korie_train[["abs_path", "text"]]],
    ignore_index=True,
)

if SUBSET_N is not None:
    df_train = df_train.sample(min(SUBSET_N, len(df_train)), random_state=0).reset_index(drop=True)
    df_val = df_val.sample(min(max(SUBSET_N // 5, 5), len(df_val)), random_state=0).reset_index(drop=True)

train_dataset = LineDataset(df_train, processor, transform=train_transform if AUGMENT else None)
val_dataset = LineDataset(df_val, processor)  # val엔 augmentation 미적용 (평가 일관성 유지)

print(f"own train: {len(df_own_train)}개(높이 필터로 {n_dropped_own}개 제외) "
      f"+ KORIE(검증 이미지 제외): {len(df_korie_train)}개(높이 필터로 {n_dropped_korie}개 제외)")
print(f"train 합계: {len(train_dataset)}개, val: {len(val_dataset)}개 | augment={AUGMENT}")


## 2. 평가지표 (CER/WER)

In [ ]:
def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)

    # 빈 참조가 있으면 jiwer가 에러를 내므로 걸러낸다
    pairs = [(r, h) for r, h in zip(label_str, pred_str) if r.strip()]
    if not pairs:
        return {"cer": 0.0, "wer": 0.0}
    refs, hyps = zip(*pairs)

    return {
        "cer": jiwer.cer(list(refs), list(hyps)),
        "wer": jiwer.wer(list(refs), list(hyps)),
    }


## 3. 파인튜닝 전 baseline 평가

In [ ]:
model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID).to(DEVICE)

# generate 설정 (Seq2SeqTrainer가 평가 시 사용)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id or processor.tokenizer.bos_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id or processor.tokenizer.eos_token_id
model.config.max_length = MAX_LABEL_LENGTH
model.config.no_repeat_ngram_size = 3


def evaluate_baseline(model, dataset, batch_size=8):
    model.eval()
    all_preds, all_refs = [], []
    with torch.no_grad():
        for i in range(0, len(dataset), batch_size):
            batch = [dataset[j] for j in range(i, min(i + batch_size, len(dataset)))]
            pixel_values = torch.stack([b["pixel_values"] for b in batch]).to(DEVICE)
            generated = model.generate(pixel_values, max_length=MAX_LABEL_LENGTH)
            preds = processor.batch_decode(generated, skip_special_tokens=True)

            labels = torch.stack([b["labels"] for b in batch])
            labels[labels == -100] = processor.tokenizer.pad_token_id
            refs = processor.batch_decode(labels, skip_special_tokens=True)

            all_preds.extend(preds)
            all_refs.extend(refs)

    pairs = [(r, h) for r, h in zip(all_refs, all_preds) if r.strip()]
    refs, hyps = zip(*pairs)
    return jiwer.cer(list(refs), list(hyps)), jiwer.wer(list(refs), list(hyps)), list(zip(all_refs, all_preds))


start = time.time()
baseline_cer, baseline_wer, baseline_samples = evaluate_baseline(model, val_dataset)
print(f"[파인튜닝 전] val CER: {baseline_cer:.4f}, WER: {baseline_wer:.4f} ({time.time()-start:.1f}sec)")
print()
print("샘플 (정답 -> 예측):")
for ref, hyp in baseline_samples[:5]:
    print(f"  '{ref}' -> '{hyp}'")


## 4. 파인튜닝

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    dataloader_persistent_workers=True,  # 에폭마다 워커를 새로 fork하지 않음 (경고 반복 방지 + 약간 더 빠름)
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    predict_with_generate=True,
    generation_max_length=MAX_LABEL_LENGTH,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    report_to=[],
    fp16=torch.cuda.is_available(),
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    processing_class=processor,
    # CER이 EARLY_STOPPING_PATIENCE(4) 에폭 연속 개선 안 되면 조기 종료.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

train_result = trainer.train()


In [ ]:
trainer.save_model(os.path.join(OUT_DIR, "best"))
processor.save_pretrained(os.path.join(OUT_DIR, "best"))
print("모델 저장 완료:", os.path.join(OUT_DIR, "best"))


## 5. 파인튜닝 후 평가 및 전후 비교

In [ ]:
finetuned_cer, finetuned_wer, finetuned_samples = evaluate_baseline(trainer.model, val_dataset)

print(f"[파인튜닝 전] val CER: {baseline_cer:.4f}, WER: {baseline_wer:.4f}")
print(f"[파인튜닝 후] val CER: {finetuned_cer:.4f}, WER: {finetuned_wer:.4f}")
print(f"CER 개선폭: {(baseline_cer - finetuned_cer) * 100:.2f}%p "
      f"({(1 - finetuned_cer / baseline_cer) * 100:.1f}% 상대 개선)" if baseline_cer > 0 else "")
print()
print("샘플 비교 (정답 / 파인튜닝 전 / 파인튜닝 후):")
for (ref, before), (_, after) in zip(baseline_samples[:8], finetuned_samples[:8]):
    print(f"  정답: {ref}")
    print(f"  전  : {before}")
    print(f"  후  : {after}")
    print()

pd.DataFrame({
    "metric": ["CER", "WER"],
    "before": [baseline_cer, baseline_wer],
    "after": [finetuned_cer, finetuned_wer],
}).to_csv(os.path.join(OUT_DIR, "before_after_comparison.csv"), index=False)


## 6. 직접 테스트해보기 (원하는 이미지로 확인)

학습이 끝난 뒤 `trocr_finetuned_v2/best`에 저장된 모델을 다시 불러와서, 원하는 크롭 이미지를 넣어보고
결과를 직접 눈으로 확인할 수 있는 셀입니다. 위쪽 학습 셀들과 독립적으로 동작하므로(커널을 새로 켜고
이 섹션만 실행해도 됨), 나중에 아무 때나 다시 실행해서 모델 상태를 점검할 수 있습니다.

- 첫 번째 셀: 모델 로드
- 두 번째 셀: 내가 원하는 이미지 경로 하나를 직접 지정해서 인식 결과 확인 (이미지도 같이 출력됨)
- 세 번째 셀: val set에서 무작위로 몇 개 뽑아서 정답과 비교(맞았는지 ✅/❌ 표시)


In [ ]:
import os

os.environ.setdefault("USE_TF", "0")

import torch
from PIL import Image
from IPython.display import display
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

PROJECT_DIR = os.getcwd()
BEST_MODEL_DIR = os.path.join(PROJECT_DIR, "trocr_finetuned_v2", "best")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

test_processor = TrOCRProcessor.from_pretrained(BEST_MODEL_DIR)
test_model = VisionEncoderDecoderModel.from_pretrained(BEST_MODEL_DIR).to(DEVICE)
test_model.eval()


def ocr_predict(image_path, max_length=64):
    """줄 단위로 잘라낸 크롭 이미지 한 장을 넣으면 인식 결과 문자열을 반환.
    (영수증 전체 사진을 통째로 넣으면 한 줄만 인식하도록 학습된 모델이라 결과가 이상하게 나옴)"""
    image = Image.open(image_path).convert("RGB")
    pixel_values = test_processor(images=image, return_tensors="pt").pixel_values.to(DEVICE)
    with torch.no_grad():
        generated = test_model.generate(pixel_values, max_length=max_length)
    return test_processor.batch_decode(generated, skip_special_tokens=True)[0]


print("모델 로드 완료:", BEST_MODEL_DIR)


In [ ]:
# 원하는 이미지 경로로 바꿔서 테스트해보세요.
# 예시로는 우리 자체 크롭 하나를 넣어둠 - own/crops, KORIE/ocr/{train,val,test}, 또는 직접 잘라낸
# 새 이미지 경로 아무거나 가능.
image_path = os.path.join(PROJECT_DIR, "DATA", "own", "crops", "IMG00004_0.png")

display(Image.open(image_path))
print("인식 결과:", ocr_predict(image_path))


In [ ]:
import random

import pandas as pd

N_SAMPLES = 5

df_val_check = pd.read_csv(os.path.join(PROJECT_DIR, "DATA", "own", "metadata_val.csv"))
sample_rows = df_val_check.sample(N_SAMPLES).reset_index(drop=True)

n_correct = 0
for _, row in sample_rows.iterrows():
    path = os.path.join(PROJECT_DIR, "DATA", "own", row["file_path"])
    pred = ocr_predict(path)
    is_correct = pred.strip() == row["text"].strip()
    n_correct += is_correct

    display(Image.open(path))
    print("✅ 일치" if is_correct else "❌ 불일치")
    print(f"  정답: {row['text']}")
    print(f"  예측: {pred}")
    print()

print(f"{N_SAMPLES}개 중 {n_correct}개 완전 일치")


### 6-1. 크롭 말고 전체 영수증으로 확인 (test/val 이미지만)

`image_id`를 지정하면 원본 영수증 전체 이미지를 보여주고, 그 영수증에 속한 줄 단위 크롭들을
순서대로(위→아래) 모델에 넣어 정답/예측을 나란히 보여줍니다.

**참고**: 이 프로젝트 v2에서 진짜 "한 번도 학습에 안 쓰인" 데이터는 우리 자체 val셋(45장)뿐입니다.
KORIE의 `ocr/test/` 폴더는 이름은 test지만 학습 데이터를 합칠 때 걸러내지 않아서 실제로는 학습에
섞여 들어갔습니다(우리 val 이미지 45장만 제외함). 그래서 아래 셀은 **val셋(=사실상 test) 45장 중에서만**
`image_id`를 무작위로 골라 보여줍니다.


In [ ]:
import random
import re

import pandas as pd

RECEIPT_DIR = os.path.join(PROJECT_DIR, "train", "images")

# 학습에 한 번도 안 쓰인 val(=test) 이미지 45장 중에서만 고른다
df_val_all = pd.read_csv(os.path.join(PROJECT_DIR, "DATA", "own", "metadata_val.csv"))
val_image_ids = sorted(set(df_val_all["file_path"].str.extract(r"(IMG\d+)")[0]))

image_id = random.choice(val_image_ids)  # 특정 이미지를 보고 싶으면 val_image_ids 중 하나로 직접 지정해도 됨
print(f"선택된 영수증(val/test, 학습 미사용): {image_id}  (전체 val 이미지 목록: {val_image_ids})")

receipt_path = os.path.join(RECEIPT_DIR, image_id + ".png")
display(Image.open(receipt_path))

rows = df_val_all[df_val_all["file_path"].str.extract(r"(IMG\d+)")[0] == image_id].copy()
rows["idx"] = rows["file_path"].str.extract(r"_(\d+)\.png$")[0].astype(int)
rows = rows.sort_values("idx")

print(f"\n{image_id} 영수증의 줄 {len(rows)}개 (정답 / 예측):\n")
for _, row in rows.iterrows():
    path = os.path.join(PROJECT_DIR, "DATA", "own", row["file_path"])
    pred = ocr_predict(path)
    mark = "✅" if pred.strip() == row["text"].strip() else "❌"
    print(f"{mark} 정답: {row['text']}")
    print(f"   예측: {pred}")


## 7. 완전히 새로운 영수증 사진으로 종합(검출+인식) 테스트

위 6번 섹션은 이미 잘라놓은 줄 크롭을 넣는 거라 "인식"만 테스트하는 거였는데, 여긴 **잘라내지 않은
새 영수증 사진 한 장을 통째로** 넣으면 EasyOCR로 줄을 검출하고, 각 줄을 우리 파인튜닝 모델로
인식까지 하는 **전체 파이프라인**을 테스트합니다. 학습/검증 데이터에 전혀 없던 사진으로 시도하면
실사용 환경에서의 진짜 성능(검출 단계 오류 포함)을 볼 수 있습니다.

- 정답(GT)이 없는 새 사진이라 CER 같은 정량 지표는 못 내고, 인식된 줄들을 눈으로 확인하는 용도입니다.
- 확인해본 결과 val CER(8%)보다 훨씬 안 좋게 나올 수 있는데, 학습 데이터와 다른 매장/폰트/사진
  조건에서는 EasyOCR 검출 자체도 흔들리기 때문입니다(인식 모델만의 문제가 아님).


In [ ]:
import easyocr

# 6번 섹션에서 이미 로드한 test_model / test_processor / DEVICE 를 그대로 재사용
reader = easyocr.Reader(["ko", "en"], gpu=torch.cuda.is_available())


def box_rect(box):
    xs = [p[0] for p in box]
    ys = [p[1] for p in box]
    return min(xs), min(ys), max(xs), max(ys)


def sort_reading_order(boxes):
    """위→아래, 같은 줄 안에서는 왼쪽→오른쪽 순으로 정렬."""
    rects = [box_rect(b) for b in boxes]
    order = sorted(range(len(boxes)), key=lambda i: (rects[i][1] + rects[i][3]) / 2)
    heights = [r[3] - r[1] for r in rects]
    avg_h = sum(heights) / len(heights)

    lines, current, current_y = [], [order[0]], (rects[order[0]][1] + rects[order[0]][3]) / 2
    for idx in order[1:]:
        yc = (rects[idx][1] + rects[idx][3]) / 2
        if abs(yc - current_y) < avg_h * 0.6:
            current.append(idx)
        else:
            lines.append(current)
            current = [idx]
            current_y = yc
    lines.append(current)

    result = []
    for line in lines:
        result.extend(sorted(line, key=lambda i: rects[i][0]))
    return [(boxes[i], rects[i]) for i in result]


def ocr_predict_image(pil_image, max_length=64):
    """이미 메모리에 있는 PIL 이미지(크롭)를 인식 - 파일로 저장 안 하고 바로 처리."""
    pixel_values = test_processor(images=pil_image, return_tensors="pt").pixel_values.to(DEVICE)
    with torch.no_grad():
        generated = test_model.generate(pixel_values, max_length=max_length)
    return test_processor.batch_decode(generated, skip_special_tokens=True)[0]


def ocr_full_receipt(image_path):
    """영수증 사진 한 장 -> (EasyOCR 검출 + 우리 모델 인식) 결과 줄 리스트."""
    pil_full = Image.open(image_path).convert("RGB")
    boxes_raw = reader.detect(image_path)[0][0]  # 박스 좌표만 사용, 인식은 우리 모델로
    if not boxes_raw:
        return []

    boxes = []
    for x_min, x_max, y_min, y_max in boxes_raw:
        boxes.append([[x_min, y_min], [x_max, y_min], [x_max, y_max], [x_min, y_max]])

    ordered = sort_reading_order(boxes)
    lines = []
    for _, rect in ordered:
        x0, y0, x1, y1 = [int(v) for v in rect]
        crop = pil_full.crop((max(0, x0), max(0, y0), x1, y1))
        if crop.width < 2 or crop.height < 2:
            continue
        lines.append(ocr_predict_image(crop))
    return lines


print("준비 완료 - 아래 셀에서 image_path만 바꿔서 실행하세요.")


In [ ]:
# 확인하고 싶은 새 영수증 사진 경로로 바꾸세요 (프로젝트 루트에 올려둔 사진 예시)
image_path = os.path.join(PROJECT_DIR, "image.png")

display(Image.open(image_path))
for line in ocr_full_receipt(image_path):
    print(" ", line)
